# Circular regression with Unwrap (R)

A minimal, fully self-contained introduction, in R, to the circular
regression machinery of `unwrap`: individual angular observations, with
both the von Mises mean direction and the concentration modeled as
functions of a continuous predictor, and all angles in **radians**. The
notebook generates synthetic angular data, fits a `brms` von Mises model
through the repository's custom `unwrap_von_mises` family
(`R/unwrap_family.R`), and checks the posterior — all in one sitting,
with no external data and no downloads.

This is the **first R notebook you should run** if you are new to the
R examples; it is also the only R notebook executed by the CI pipeline
(`.github/workflows/notebooks.yml`), with shortened sampling.

The Python version of this tutorial
(`notebooks/circular_regression.ipynb`) fits the same model (same
data-generating parameters, links, and prior scales) with Bambi and PyMC.

## Setup

There is no one-click R runtime behind these notebooks (Colab is
Python-only), so set the R side up once: the [R setup](../getting-started-r.md)
page installs the CRAN packages, CmdStan, and the Jupyter R kernel. The
first code cell below re-checks the environment and restores missing
pieces, so on a set-up machine it is a no-op.

In [ ]:
# Setup: make sure this notebook's stack is installed. The Python
# notebooks rely on the local virtual environment (or Colab's bootstrap
# cell); R has no one-click runtime, so this cell guarantees the packages
# instead. Follow the R setup page for the manual route.
# cmdstanr is distributed by the Stan team on r-universe (it is no
# longer on CRAN); everything else comes from CRAN.
if (!requireNamespace("cmdstanr", quietly = TRUE)) {
  # two repos so cmdstanr's CRAN dependencies resolve as well
  install.packages("cmdstanr",
    repos = c("https://stan-dev.r-universe.dev", "https://cloud.r-project.org"))
}
required <- c("brms", "posterior", "loo", "circular", "ggplot2")
missing <- setdiff(required, rownames(installed.packages()))
if (length(missing)) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}
cmdstan_ok <- tryCatch(dir.exists(as.character(cmdstanr::cmdstan_path())),
  error = function(e) FALSE)
if (!cmdstan_ok) {
  # One-time ~50 MB download (see the R setup page).
  cmdstanr::install_cmdstan()
}

In [ ]:
suppressPackageStartupMessages({
    library(brms)
    library(posterior)
    library(loo)
    library(circular)
    library(ggplot2)
})

# The repository's R helpers (custom brms family + circular statistics)
# live next to the notebook; make them reachable regardless of the
# kernel's working directory.
repo_root <- if (file.exists(file.path("..", "R", "unwrap_family.R"))) ".." else
    if (file.exists(file.path("R", "unwrap_family.R"))) "." else
    stop("Run this notebook from notebooks_r/ or from the repository root.")
source(file.path(repo_root, "R", "unwrap_family.R"))
source(file.path(repo_root, "R", "unwrap_functions.R"))

# The figure code writes to the repository-relative figures directory.
FIG_DIR <- file.path(repo_root, "figures", "circular_regression")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
theme_set(theme_minimal(base_size = 12))

SEED <- 42
# CI runs with shortened sampling (the committed outputs use the full
# budget); local runs always use the full budget.
ci_run <- Sys.getenv("UNWRAP_R_CI") == "1"
CHAINS <- if (ci_run) 2L else 4L
ITER <- if (ci_run) 800L else 1000L
WARMUP <- if (ci_run) 400L else 500L

c(R = as.character(getRversion()),
  brms = as.character(packageVersion("brms")),
  cmdstanr = as.character(packageVersion("cmdstanr")),
  posterior = as.character(packageVersion("posterior")),
  loo = as.character(packageVersion("loo")),
  circular = as.character(packageVersion("circular")),
  ggplot2 = as.character(packageVersion("ggplot2")))

## Figure style

One figure style for every figure (the shared house convention): figures
render inline and are saved as vector (PDF) files under
`figures/circular_regression/` (gitignored).

## 1. Generate synthetic angular data

Draw from a von Mises model with known mean and concentration — a
test-bed: because the true relationship is known, we can check that the
posterior recovers it. For predictor $x \in (-1, 1)$, the generating
mean is $\mu = x$ (wrapped to $[-\pi,\pi)$) and the concentration is
$\kappa = \operatorname{softplus}(2 + 0.5x)$: larger concentration
means less angular dispersion. These values create a simple example;
they do not prescribe priors for a real dataset. Sampling uses
`circular::rvonmises`, the same distribution PyMC draws from in the
Python tutorial.

In [ ]:
set.seed(SEED)
n_samples <- 200
x <- runif(n_samples, min = -1, max = 1)

# Generating model (known ground truth): mu = 0 + 1 * x wrapped, and
# kappa = softplus(2 + 0.5 * x) - the concentration on the natural
# scale; the model below recovers the pre-softplus values (2 and 0.5).
mu_true <- mod_circular(x)
kappa_true <- softplus(2 + 0.5 * x)
# circular 0.5.x samplers take scalar mu and kappa plus a control list;
# draw on the default [0, 2*pi) domain and wrap back to the model's
# (-pi, pi] convention. (These helpers are reused in the PPC cell below.)
vmu <- function(m) circular::circular(
    m, units = "radians", template = "none", modulo = "2pi")
vm_ctrl <- list(units = "radians", template = "none", modulo = "2pi")
y <- vapply(
    seq_len(n_samples),
    function(i) as.numeric(circular::rvonmises(1,
        mu = vmu(mu_true[i]),
        kappa = kappa_true[i],
        control.circular = vm_ctrl)),
    numeric(1))
y <- mod_circular(y)

data <- data.frame(x = x, y = y)
stopifnot(all(data$y <= pi & data$y >= -pi))
head(data)

## 2. Specify the model

`brms`'s formula interface describes the regression; the circular
nature is carried by the custom `unwrap_von_mises` family from
`R/unwrap_family.R`, whose Stan log-density applies the circular modulo
link to the mean *inside* the likelihood.

`y ~ x` describes the mean direction; `kappa ~ x` describes
concentration. Softplus keeps the concentration positive, so the
concentration coefficients live on the **pre-softplus** scale (the
Python tutorial puts its priors on the same scale). The mean intercept
has an approximately uniform circular prior; the slope priors assume
this predictor's scale of $-1$ to $1$. Adapt these choices to your
scientific question and check prior predictive behavior when analyzing
real data.

In [ ]:
# y ~ x: mean direction; kappa ~ x: concentration
form_vonmises <- bf(y ~ x, kappa ~ x, family = unwrap_von_mises)

prior_vonmises <- (
  # approximately uniform circular prior on the mean intercept
  prior("unwrap_von_mises(0, 1e-16)", class = "Intercept", dpar = "mu") +
  # slope priors assume this predictor's scale of -1 to 1
  prior("normal(0, pi()/2)", class = "b", dpar = "mu") +
  prior("normal(2.0, 0.5)", class = "Intercept", dpar = "kappa") +
  prior("normal(0.0, 1.0)", class = "b", dpar = "kappa")
)
prior_vonmises

## 3. Fit and inspect sampling

`brm()` runs NUTS via Stan (the `cmdstanr` backend). Four chains with
500 warmup iterations and 500 retained draws each keep this
introduction manageable — the same sampling budget as the Python
tutorial — and they are not a universal sampling prescription.

Inspect the divergence count first — $\hat{R}$ and the effective
sample sizes are only meaningful when it is 0. Not every R backend
build keeps the per-iteration divergence flag in the fit object, in
which case the count is reported accordingly and the $\hat{R}$ and ESS
checks below carry the convergence evidence.
$\hat{R} \approx 1$ (computed on the *unwrapped* intercept: the
repository's `Rhat_unwrap` centres the angular draws before comparing
chains, the circular analogue of the naive R-hat) and effective sample
sizes above 400. The true mean direction used to generate the data
should be recoverable from the posterior; increase sampling or revise
the model if the diagnostics are unsatisfactory. CI checks execution,
not parameter recovery or convergence thresholds.

In [ ]:
fit <- brm(
    form_vonmises, data = data,
    stanvars = stan_unwrap_fun + modulo_circular_fun,
    prior = prior_vonmises,
    chains = CHAINS, iter = ITER, warmup = WARMUP, cores = CHAINS,
    seed = SEED, control = list(adapt_delta = 0.9),
    backend = "cmdstanr")
# NUTS divergences. Not every brms/cmdstanr combination keeps the
# per-iteration divergence flag in the fit object, so probe for it and
# report what the fit object actually carries (the R-hat and ESS values
# below are the primary convergence checks either way).
divergences <- tryCatch({
  div_flags <- lapply(fit$fit@sim$samples, function(ch) ch[["divergent"]])
  if (any(vapply(div_flags, is.null, logical(1)))) NA_real_
  else sum(unlist(div_flags))
}, error = function(e) NA_real_)
if (is.na(divergences)) {
    cat("Divergences: not retained in this fit object (see R-hat/ESS below).\n")
} else {
    cat("Divergences:", divergences, "\n")
}

parameters <- c("b_Intercept", "b_x", "kappa_Intercept", "kappa_x")
draws <- posterior::as_draws_df(fit, variables = parameters)
stopifnot(all(vapply(draws, all, FUN = is.finite)))

# Convergence: R-hat (unwrapped for the circular intercept) and ESS
rhat_values <- c(
    b_Intercept = Rhat_unwrap(draws$b_Intercept, method = "mean"),
    b_x = posterior::rhat(draws$b_x),
    kappa_Intercept = posterior::rhat(draws$kappa_Intercept),
    kappa_x = posterior::rhat(draws$kappa_x))
ess_values <- vapply(parameters, function(p) posterior::ess(draws[[p]]), numeric(1))

# Posterior summary. The intercept is circular, so its location is the
# circular mean and its spread the circular standard deviation - the
# R analogue of the Python tutorial's circular_summary().
circ_mean <- function(a) atan2(mean(sin(a)), mean(cos(a)))
circ_sd <- function(a, m) sqrt(-2 * log(Mod(mean(exp(1i * (a - m))))))
m_int <- circ_mean(draws$b_Intercept)
summary_frame <- data.frame(
    parameter = parameters,
    mean = c(m_int, mean(draws$b_x), mean(draws$kappa_Intercept),
             mean(draws$kappa_x)),
    sd = c(circ_sd(draws$b_Intercept, m_int), sd(draws$b_x),
           sd(draws$kappa_Intercept), sd(draws$kappa_x)),
    rhat = round(rhat_values, 4),
    ess = round(ess_values, 1),
    # generating values (kappa on the pre-softplus scale)
    truth = c(0, 1, 2, 0.5),
    check.names = FALSE)
summary_frame

In [ ]:
# Posterior traces for the four parameters (the figure is saved under
# figures/circular_regression/).
trace_df <- posterior::as_draws_df(fit, variables = parameters)
p_trace <- ggplot(trace_df, aes(x = value, y = .chain, colour = parameter)) +
    geom_line(alpha = 0.4) +
    scale_y_continuous(breaks = sort(unique(trace_df$.chain)) - 0.5,
                       expand = c(0.05, 0.05)) +
    labs(x = "", y = "chain") +
    theme(legend.position = "bottom")
ggsave(file.path(FIG_DIR, "trace.pdf"), p_trace, width = 7, height = 3.5)
p_trace

## 4. Interpret predictions on the circle

The prediction cell evaluates the von Mises mean direction and
concentration across a range of the predictor — the direct analogue of
a standard model-averaged prediction curve, but drawn on the circle.

Average predicted directions as unit vectors, not ordinary numbers:
angles near $-\pi$ and $\pi$ are neighbours, so the mean-direction
panel uses the circular mean of each grid point's posterior draws while
the second panel averages the (post-softplus) concentration draws.
These curves summarize the posterior; they are not uncertainty
intervals. The dashed generating curves are available only because this
dataset is simulated.

In [ ]:
grid_x <- seq(-1, 1, length.out = 100)

# Posterior draws of the linear predictors: one row per grid point, one
# column per draw.
lin_mu <- outer(grid_x, draws$b_x,
                function(g, b) draws$b_Intercept + g * b)
lin_kappa <- outer(grid_x, draws$kappa_x,
                   function(g, b) draws$kappa_Intercept + g * b)

# Average the predicted directions as unit vectors (angles near +/-pi
# are neighbours), and average the concentration draws for the right-
# hand panel.
mean_direction <- mapply(function(l) atan2(mean(sin(l)), mean(cos(l))),
                         seq_len(ncol(lin_mu)))
concentration <- apply(lin_kappa, 2, function(l) mean(softplus(l)))

obs_df <- data.frame(grid = data$x, value = data$y,
                     panel = "mean direction (rad)")
pred_df <- data.frame(
    grid = rep(grid_x, 2),
    value = c(mean_direction, concentration),
    panel = c(rep("mean direction (rad)", length(grid_x)),
              rep("concentration (kappa)", length(grid_x))))
p_pred <- ggplot(pred_df, aes(grid, value)) +
    geom_line(alpha = 0.8) +
    geom_point(data = obs_df, aes(grid, value),
               alpha = 0.35, position = position_jitter(width = 0.02, height = 0.15)) +
    geom_line(data = data.frame(grid = grid_x,
                                value = mod_circular(grid_x),
                                panel = "mean direction (rad)"),
              inherit.aes = FALSE, linetype = "dashed") +
    geom_line(data = data.frame(grid = grid_x,
                                value = softplus(2 + 0.5 * grid_x),
                                panel = "concentration (kappa)"),
              inherit.aes = FALSE, linetype = "dashed") +
    facet_wrap(~ panel, scales = "free_y") +
    labs(x = expression(x), y = "")
ggsave(file.path(FIG_DIR, "predictions.pdf"), p_pred, width = 8, height = 3)
p_pred

## 5. Check posterior predictive behavior

Simulate new angles at the observed predictor values and compare the
observed mean resultant length (a circular concentration summary) with
that statistic in the replicated datasets. For a well-specified model
the two should match; a systematic mismatch (the predictive too narrow,
or the predicted mean shifted) signals misspecification, as in a
standard Gaussian GLM. This is one aggregate check: it can miss local
lack of fit and reflects variation in both mean direction and
concentration across $x$. For real data, add checks conditional on
predictors and relevant groups.

The fitted links are applied explicitly below (the modulo link for the
mean, softplus for the concentration) and the replicates are drawn with
`circular::rvonmises` — the same von Mises sampler used to generate the
data.

In [ ]:
obs_x <- data$x
lin_mu_obs <- draws$b_Intercept + draws$b_x * obs_x            # draw x obs
lin_kappa_obs <- draws$kappa_Intercept + draws$kappa_x * obs_x

# The 0.5.x sampler draws one observation at a time (scalar mu/kappa),
# so replicate in a draw-by-observation nested loop (~2-4 min for the
# full posterior here).
replicated <- matrix(0, nrow = nrow(lin_mu_obs), ncol = length(obs_x))
for (d in seq_len(nrow(lin_mu_obs)))
  for (i in seq_len(length(obs_x)))
    replicated[d, i] <- as.numeric(circular::rvonmises(1,
        mu = vmu(lin_mu_obs[d, i]),
        kappa = softplus(lin_kappa_obs[d, i]),
        control.circular = vm_ctrl))
replicated <- mod_circular(replicated)
# Replicate the data at the observed x values; the figure below compares
# the observed mean resultant length against the replicated one.
replicated_R <- apply(exp(1i * replicated), 1, function(z) Mod(mean(z)))
observed_R <- Mod(mean(exp(1i * data$y)))
stopifnot(all(is.finite(replicated_R)))

p_ppc <- ggplot(data.frame(replicated_R), aes(replicated_R)) +
    geom_density(alpha = 0.7, fill = "darkseagreen", colour = "grey40") +
    geom_vline(xintercept = observed_R, colour = "firebrick", linewidth = 1) +
    labs(x = "mean resultant length R", y = "density")
ggsave(file.path(FIG_DIR, "ppc.pdf"), p_ppc, width = 5, height = 3)
p_ppc

## Next steps

The beetle, zebra, and locust notebooks explore more complex analyses
and may require additional setup. Start by defining the scientific
estimand, angular reference direction, and sampling structure. This
tutorial assumes independent observations and does not address
hierarchical dependence or mixtures.

Running all cells records the installed dependency versions and
regenerates the figures. The Python version of this tutorial lives at
`notebooks/circular_regression.ipynb`; the standalone R scripts
(`R/brms_example.R`, `R/unwrap_functions.R`) that this notebook builds
on are documented on the [R and MATLAB](../languages.md) page.